In [1]:
import sys
!{sys.executable} -m pip install ydata-profiling

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 3.4 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd


df = pd.read_csv('/content/drive/MyDrive/FDE Projects/FinSight/finsight_loan_applications.csv')
print(df.shape)
print(df.columns.tolist())
df.head(3)


/tmp/ipykernel_12108/3458729343.py:2: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


(130, 13)
['loan_id', 'application_date', 'applicant_age', 'employment_status', 'annual_income', 'loan_amount', 'loan_tenure_months', 'credit_score', 'existing_emis', 'loan_purpose', 'branch', 'loan_status', 'default_flag']


,loan_id,application_date,applicant_age,employment_status,annual_income,loan_amount,loan_tenure_months,credit_score,existing_emis,loan_purpose,branch,loan_status,default_flag
0,LN00001,2024-01-23,47,Retired,NaN,382358,48,498.0,4,Business,Hyderabad,Defaulted,1
1,LN00002,2024-09-17,48,Unemployed,NaN,1285279,60,782.0,4,Business,Hyderabad,Closed,0
2,LN00003,2024-06-22,36,Retired,NaN,1158273,36,521.0,4,Home Renovation,Delhi,Rejected,0


In [5]:
from ydata_profiling import ProfileReport

profile = ProfileReport(df, title='FinSight Loan Application Audit', explorative=True)
profile.to_file('/content/drive/MyDrive/FDE Projects/FinSight/finsight_loan_audit.html')
print("Report saved.")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 13/13 [00:00<00:00, 92.60it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Report saved.


# **Part A — Data Profiling & Risk Data Audit**
The credit team says the data is 'pulled straight from the system'. Profile it before touching any modelling.

1.	Load finsight_loan_applications.csv into a Pandas DataFrame.
2.	Run ydata-profiling titled 'FinSight Loan Application Audit'. Save the HTML report.
3.	From the report, document in a markdown cell:
•	How many rows are missing annual_income, and which employment_status categories are affected most?
•	How many rows are missing credit_score? Why is this particularly dangerous for a default prediction model?
•	What is the default_flag distribution (% defaulted)? Is the dataset imbalanced?
•	Are there any loan_amount values that appear statistically implausible (e.g. below 10,000 or above 5,000,000)?
4.	Write two sentences for the Head of Credit Risk: (a) which missing column is the most dangerous gap, and (b) what the bank risks if the model is trained on incomplete data.


In [6]:
# Missing annual_income
income_missing = df['annual_income'].isnull().sum()
income_missing_by_employment = df[df['annual_income'].isnull()]['employment_status'].value_counts()

# Missing credit_score
credit_missing = df['credit_score'].isnull().sum()

# Default flag distribution
default_dist = df['default_flag'].value_counts(normalize=True).round(3) * 100

# Implausible loan amounts
implausible_low = df[df['loan_amount'] < 10000].shape[0]
implausible_high = df[df['loan_amount'] > 5000000].shape[0]

print("=== MISSING annual_income ===")
print(f"Total missing: {income_missing}")
print(income_missing_by_employment)

print("\n=== MISSING credit_score ===")
print(f"Total missing: {credit_missing}")

print("\n=== DEFAULT FLAG DISTRIBUTION ===")
print(default_dist)

print("\n=== IMPLAUSIBLE LOAN AMOUNTS ===")
print(f"Below 10,000: {implausible_low}")
print(f"Above 50,00,000: {implausible_high}")

=== MISSING annual_income ===
Total missing: 5
employment_status
Retired          2
Unemployed       2
Self-Employed    1
Name: count, dtype: int64

=== MISSING credit_score ===
Total missing: 6

=== DEFAULT FLAG DISTRIBUTION ===
default_flag
0    72.3
1    27.7
Name: proportion, dtype: float64

=== IMPLAUSIBLE LOAN AMOUNTS ===
Below 10,000: 0
Above 50,00,000: 0


## Part A — Data Audit Findings

### Missing Data
- **annual_income**: 5 rows missing. Employment categories affected: Retired (2), Unemployed (2), Self-Employed (1).
- **credit_score**: 6 rows missing. This is the most dangerous gap — a missing bureau score means the RBI-mandated credit check was never completed for those applicants.

### Default Flag Distribution
- 27.7% of applications defaulted (default_flag = 1). Dataset is moderately imbalanced.

### Loan Amount Plausibility
- No values below ₹10,000 or above ₹50,00,000. Loan amounts pass the plausibility check.

### Brief for Head of Credit Risk
(a) The most dangerous missing field is **credit_score** — 6 applications have no bureau score on record, meaning approval decisions for those loans were made without a mandatory risk check.
(b) If the model is trained on incomplete data, it will learn from a distorted picture of risk — applicants who appear low-risk only because their data is missing, leading the bank to approve loans it should flag.